# Pandas va Sklearn: Gipotezalarni tekshirish va Unknown Values bilan ishlash

Bu notebookda:

1. `sklearn.datasets`dan dataset topamiz va statistik **gipotezani** t-test yordamida tekshiramiz
2. Haqiqiy `ML_Engineeer.csv` dataseti ustida ham gipoteza tekshiramiz
3. **Unknown / missing values**ni sun'iy yaratib, ularni aniqlash va tozalash usullarini ko'ramiz

---

## 1. Kutubxonalarni import qilish

In [1]:
import pandas as pd
import numpy as np
from scipy import stats
from sklearn.datasets import load_diabetes

pd.set_option('display.max_columns', 20)

## 2. Sklearn'dan dataset topish: Diabetes dataseti

`load_diabetes()` — 442 nafar bemorning yoshi, jinsi, BMI, qon bosimi va boshqa ko'rsatkichlarini o'z ichiga oladi. Maqsad — kasallik progressini (`target`) bashorat qilish.

In [2]:
data = load_diabetes()
diab_df = pd.DataFrame(data.data, columns=data.feature_names)
diab_df['target'] = data.target

print(diab_df.shape)
diab_df.head()

(442, 11)


,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6,target
0,0.038076,0.050680,0.061696,0.021872,-0.044223,-0.034821,-0.043401,-0.002592,0.019907,-0.017646,151.0
1,-0.001882,-0.044642,-0.051474,-0.026328,-0.008449,-0.019163,0.074412,-0.039493,-0.068332,-0.092204,75.0
2,0.085299,0.050680,0.044451,-0.005670,-0.045599,-0.034194,-0.032356,-0.002592,0.002861,-0.025930,141.0
3,-0.089063,-0.044642,-0.011595,-0.036656,0.012191,0.024991,-0.036038,0.034309,0.022688,-0.009362,206.0
4,0.005383,-0.044642,-0.036385,0.021872,0.003935,0.015596,0.008142,-0.002592,-0.031988,-0.046641,135.0


## 3. Gipotezani shakllantirish (Diabetes dataseti)

**Savol:** BMI (tana vazni indeksi) o'rtacha darajadan yuqori va past bo'lgan bemorlar orasida kasallik progressi (`target`) statistik jihatdan farq qiladimi?

* **H0:** Ikki guruh o'rtasida `target` bo'yicha farq yo'q
* **H1:** Ikki guruh o'rtasida statistik ahamiyatli farq bor

In [3]:
bmi_median = diab_df['bmi'].median()

high_bmi = diab_df[diab_df['bmi'] > bmi_median]['target']
low_bmi = diab_df[diab_df['bmi'] <= bmi_median]['target']

t_stat, p_value = stats.ttest_ind(high_bmi, low_bmi, equal_var=False)

print(f'Yuqori BMI guruhi o\'rtachasi: {high_bmi.mean():.2f}')
print(f'Past BMI guruhi o\'rtachasi: {low_bmi.mean():.2f}')
print(f't-statistic: {t_stat:.4f}')
print(f'p-value: {p_value:.10f}')

alpha = 0.05
if p_value < alpha:
    print('H0 rad etildi: BMI darajasi kasallik progressiga statistik ta\'sir qiladi.')
else:
    print('H0 rad etilmadi: statistik ahamiyatli farq topilmadi.')

Yuqori BMI guruhi o'rtachasi: 191.46
Past BMI guruhi o'rtachasi: 113.52
t-statistic: 12.2750
p-value: 0.0000000000
H0 rad etildi: BMI darajasi kasallik progressiga statistik ta'sir qiladi.


---

## 4. Haqiqiy dataset ustida gipoteza: ML Engineer maoshlari

**Savol:** To'liq shtat (`FT`) bo'yicha ishlaydiganlar va boshqa ish turlaridagilar (`CT`, `FL`, `PT`) o'rtasida maosh (`salary_in_usd`) statistik jihatdan farq qiladimi?

* **H0:** Ikki guruh o'rtasida maosh farqi yo'q
* **H1:** Ikki guruh o'rtasida statistik ahamiyatli farq bor

In [4]:
ml_df = pd.read_csv('../Data/ML_Engineeer.csv')

ft_salary = ml_df[ml_df['employment_type'] == 'FT']['salary_in_usd']
other_salary = ml_df[ml_df['employment_type'] != 'FT']['salary_in_usd']

print('FT guruhi soni:', len(ft_salary))
print('Boshqa guruh soni:', len(other_salary))

t_stat, p_value = stats.ttest_ind(ft_salary, other_salary, equal_var=False)

print(f't-statistic: {t_stat:.4f}')
print(f'p-value: {p_value:.4f}')

if p_value < alpha:
    print('H0 rad etildi: ish turi maoshga statistik ta\'sir qiladi.')
else:
    print('H0 rad etilmadi: statistik ahamiyatli farq topilmadi.')

FT guruhi soni: 4135
Boshqa guruh soni: 9
t-statistic: 3.3047
p-value: 0.0107
H0 rad etildi: ish turi maoshga statistik ta'sir qiladi.


---

## 5. Unknown / Missing Values yaratish va aniqlash

`ML_Engineeer.csv` dasetida bo'sh qiymatlar yo'q, shuning uchun buni ko'rsatish uchun sun'iy ravishda `NaN` va `'unknown'` qiymatlar qo'shamiz.

In [5]:
ml_missing = ml_df.copy()

np.random.seed(42)
rows_salary = ml_missing.sample(frac=0.04, random_state=42).index
ml_missing.loc[rows_salary, 'salary_in_usd'] = np.nan

rows_size = ml_missing.sample(frac=0.03, random_state=7).index
ml_missing.loc[rows_size, 'company_size'] = 'unknown'

ml_missing[['salary_in_usd', 'company_size']].head(10)

,salary_in_usd,company_size
0,296300.0,M
1,166600.0,M
2,264200.0,M
3,143100.0,M
4,55555.0,M
5,55555.0,M
6,297000.0,M
7,168300.0,M
8,NaN,M
9,139000.0,M


### 5.1. Noma'lum belgilarni haqiqiy NaN'ga aylantirish

In [6]:
ml_missing.replace('unknown', np.nan, inplace=True)

missing_percent = ml_missing.isnull().mean() * 100
missing_percent[missing_percent > 0]

salary_in_usd    4.005792
company_size     2.992278
dtype: float64

### 5.2. Missing qiymatlarni tozalash usullari

1. **dropna** — qatorlarni o'chirish
2. **fillna (median)** — sonli ustunlar uchun
3. **fillna (mode)** — kategoriyal ustunlar uchun

In [7]:
ml_dropped = ml_missing.dropna(subset=['salary_in_usd', 'company_size'])
print("Dropna'dan keyingi shakl:", ml_dropped.shape, '| Asl shakl:', ml_missing.shape)

Dropna'dan keyingi shakl: (3861, 11) | Asl shakl: (4144, 11)


In [8]:
ml_filled = ml_missing.copy()

ml_filled['salary_in_usd'] = ml_filled['salary_in_usd'].fillna(ml_filled['salary_in_usd'].median())
ml_filled['company_size'] = ml_filled['company_size'].fillna(ml_filled['company_size'].mode()[0])

print(ml_filled[['salary_in_usd', 'company_size']].isnull().sum())

salary_in_usd    0
company_size     0
dtype: int64


### 5.3. To'ldirishdan oldin va keyin o'rtachani solishtirish

In [9]:
print(f"Asl salary_in_usd o'rtachasi: {ml_df['salary_in_usd'].mean():.2f}")
print(f"To'ldirilgandan keyingi o'rtacha: {ml_filled['salary_in_usd'].mean():.2f}")

Asl salary_in_usd o'rtachasi: 197720.98
To'ldirilgandan keyingi o'rtacha: 197714.66


---

## 6. Xulosa

* Gipotezalarni tekshirishda **t-test** yordamida ikki guruh o'rtasidagi farq statistik ahamiyatga ega yoki emasligini aniqlaymiz (p-value < 0.05 bo'lsa H0 rad etiladi).
* Unknown qiymatlar (`'?'`, `'unknown'` va h.k.) avval haqiqiy `NaN`ga aylantirilishi kerak, so'ng `dropna` yoki `fillna` orqali tozalanadi.
* Median bilan to'ldirish odatda o'rtacha qiymatni sezilarli o'zgartirmaydi, shu bilan birga ma'lumot yo'qotilmaydi.